In [ ]:
import cv2
import mediapipe as mp
import math
from pycaw.pycaw import AudioUtilities
from pycaw.pycaw import IAudioEndpointVolume

# Mediapipe
mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils

# Cam
cap = cv2.VideoCapture(0)

#Defination of devices
devices = AudioUtilities.GetSpeakers()
interface = devices.Activate(
    IAudioEndpointVolume._iid_, 
    1, 
    None
)
volume = interface.QueryInterface(IAudioEndpointVolume)

# volume chancing
def set_volume(distance):
    # Distance is normazitaion
    if distance > 25:
        volume_level = min(max(distance - 26, 0), 200)  # Distance Max 200
        volume.SetMasterVolumeLevelScalar(volume_level / 200.0 , None)
    else :
        volume.SetMasterVolumeLevelScalar(0.0 , None)
with mp_hands.Hands(static_image_mode=False, max_num_hands=1, min_detection_confidence=0.5) as hands:
    while True:
        success, frame = cap.read()
        if not success:
            break
        
        frame = cv2.flip(frame,1)
        # BGR to RGB
        img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = hands.process(img_rgb)

        if results.multi_hand_landmarks:
            for hand_landmarks in results.multi_hand_landmarks:
                h, w, _ = frame.shape
                # Defination of fingers
                thumb_tip = hand_landmarks.landmark[4]
                index_tip = hand_landmarks.landmark[8]

                # Calculating the distance
                x1, y1 = int(thumb_tip.x * w), int(thumb_tip.y * h)
                x2, y2 = int(index_tip.x * w), int(index_tip.y * h)

                
                distance = int(math.hypot(x2 - x1, y2 - y1))

                wrist = hand_landmarks.landmark[0]
                middle_tip = hand_landmarks.landmark[12]
                z_distance = math.hypot((middle_tip.x - wrist.x) * w,(middle_tip.y - wrist.y) * h)
                print(z_distance)

                
                thumb_vector = (x1 - x2, y1 - y2)
                
               
                angle = math.atan2(y2 - y1, x2 - x1) * 180 / math.pi
                
                
                if angle < 0:
                    angle += 180  
                corrected_distance = distance * (1 + angle / 180)  #Distance calculating based on angle
                ref_scale = 230  # Deneysel olarak ayarlanabilir
                normalized_distance = corrected_distance / (z_distance / ref_scale)

                set_volume(normalized_distance)

                # Drawing
                cv2.line(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
                cx, cy = (x1 + x2) // 2, (y1 + y2) // 2
                cv2.putText(frame, f"{int(normalized_distance)} px", (cx, cy), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 0, 0), 2)
                

                
                mp_drawing.draw_landmarks(frame, hand_landmarks, mp_hands.HAND_CONNECTIONS)

        cv2.imshow("Volume Changer", frame)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()